# 🐚 Mini ML Challenge — Abalone Age Prediction

## Objective

You are working as a **Data Scientist for a marine research organization**.

The organization wants to estimate the age of an abalone from its physical measurements, without manually counting its rings.

### Challenge Question

> **Can we predict the number of rings of an abalone using its physical characteristics?**

You will follow the complete Machine Learning workflow:

**Problem Definition → Data Collection → Data Understanding → EDA → Preprocessing → Train/Test Split → Model Building → Evaluation → Model Comparison → Prediction → Conclusion**

---

## Dataset

**UCI Abalone Dataset**

Original source:

https://archive.ics.uci.edu/dataset/1/abalone

The dataset contains:

- Sex
- Length
- Diameter
- Height
- Whole weight
- Shucked weight
- Viscera weight
- Shell weight
- Rings

### Important

`Rings` is the target variable.

A commonly used approximation for age is:

**Estimated Age ≈ Rings + 1.5 years**

> The model in this project predicts **Rings**. The estimated age is calculated afterward.

# 🎯 Learning Outcomes

By completing this notebook, you should be able to:

1. Identify a Machine Learning problem as regression or classification.
2. Load and inspect a real-world dataset.
3. Perform basic exploratory data analysis.
4. Identify numerical and categorical features.
5. Encode categorical variables.
6. Split data into training and testing sets.
7. Build a Linear Regression model.
8. Build a tree-based regression model.
9. Evaluate regression models using MAE, MSE, RMSE, and R².
10. Compare two models.
11. Make a prediction for a new abalone.
12. Explain the limitations of a Machine Learning model.

# 🧩 Task 1 — Understand the Problem

Before running the Machine Learning model, answer these questions:

1. What are we trying to predict?
2. What is the target variable?
3. What are the input features?
4. Is this Classification or Regression?
5. Why?

### Your answer

Write your answers in the cell below before continuing.

In [ ]:
# YOUR ANSWERS
# 1. What are we trying to predict?
# Answer:

# 2. What is the target variable?
# Answer:

# 3. What are the input features?
# Answer:

# 4. Classification or Regression?
# Answer:

# 5. Why?
# Answer:

# 1. Import Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import warnings
warnings.filterwarnings("ignore")

# 2. Load the Dataset

The original dataset is provided by the **UCI Machine Learning Repository**.

This notebook downloads the original ZIP file directly from UCI.

In [ ]:
import zipfile
import io
import urllib.request

url = "https://archive.ics.uci.edu/static/public/1/abalone.zip"

with urllib.request.urlopen(url) as response:
    zip_data = response.read()

with zipfile.ZipFile(io.BytesIO(zip_data)) as z:
    print("Files in dataset:")
    print(z.namelist())

    # Read the CSV file from the ZIP
    with z.open("abalone.data") as f:
        df = pd.read_csv(f, header=None)

print("Dataset loaded successfully!")
df.head()

# 3. Add Column Names

The original UCI file does not contain column names, so we add them based on the official dataset description.

In [ ]:
columns = [
    "Sex",
    "Length",
    "Diameter",
    "Height",
    "WholeWeight",
    "ShuckedWeight",
    "VisceraWeight",
    "ShellWeight",
    "Rings"
]

df.columns = columns

df.head()

# 4. Understand the Dataset

Inspect:

- Number of rows
- Number of columns
- Data types
- Sample records

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nFirst 5 rows:")
display(df.head())

# 5. Data Dictionary

| Column | Description |
|---|---|
| Sex | Sex of the abalone: M, F or I |
| Length | Longest shell measurement |
| Diameter | Diameter perpendicular to length |
| Height | Height with meat in shell |
| WholeWeight | Whole abalone weight |
| ShuckedWeight | Weight of meat |
| VisceraWeight | Gut weight after bleeding |
| ShellWeight | Weight of dried shell |
| Rings | Number of rings |

### Target

`Rings`

### Features

All other useful columns.

# 6. Check Missing Values

In [ ]:
missing = df.isnull().sum()
print(missing)

print("\nTotal missing values:", df.isnull().sum().sum())

# 7. Check Duplicate Records

In [ ]:
print("Number of duplicate rows:", df.duplicated().sum())

# 8. Statistical Summary

Use `describe()` to understand the numerical variables.

Look at:

- Mean
- Standard deviation
- Minimum
- Maximum
- Quartiles

In [ ]:
df.describe()

# 9. Explore the Categorical Variable

The `Sex` column contains categorical values.

Check the number of observations in each category.

In [ ]:
df["Sex"].value_counts()

In [ ]:
df["Sex"].value_counts().plot(kind="bar")
plt.title("Distribution of Abalone Sex")
plt.xlabel("Sex")
plt.ylabel("Count")
plt.show()

# 10. Explore the Target Variable — Rings

Before building a model, understand the target variable.

### Questions

- What is the most common number of rings?
- What is the minimum?
- What is the maximum?
- Is the distribution balanced?

In [ ]:
print("Minimum Rings:", df["Rings"].min())
print("Maximum Rings:", df["Rings"].max())
print("Mean Rings:", df["Rings"].mean())
print("Median Rings:", df["Rings"].median())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df["Rings"], bins=20)
plt.title("Distribution of Rings")
plt.xlabel("Number of Rings")
plt.ylabel("Frequency")
plt.show()

# 11. Feature Relationship — Length vs Rings

Explore whether larger abalones tend to have more rings.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["Length"], df["Rings"], alpha=0.5)
plt.title("Length vs Rings")
plt.xlabel("Length")
plt.ylabel("Rings")
plt.show()

# 12. Feature Relationship — Shell Weight vs Rings

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["ShellWeight"], df["Rings"], alpha=0.5)
plt.title("Shell Weight vs Rings")
plt.xlabel("Shell Weight")
plt.ylabel("Rings")
plt.show()

# 13. Correlation Analysis

Correlation can help us understand relationships between numerical variables.

> Remember: correlation does not automatically mean causation.

In [ ]:
numeric_df = df.select_dtypes(include=np.number)
correlation = numeric_df.corr()

correlation["Rings"].sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(10, 7))
plt.imshow(correlation, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlation.columns)), correlation.columns, rotation=90)
plt.yticks(range(len(correlation.columns)), correlation.columns)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

# 🛠️ Task 2 — Data Preparation

We need to prepare the data for Machine Learning.

### Steps

1. Separate features and target.
2. Identify categorical and numerical columns.
3. Encode `Sex`.
4. Split data into training and testing sets.

### Think First

Why can't a basic Machine Learning algorithm directly use values such as `M`, `F`, and `I`?

Write your answer below.

In [ ]:
# YOUR ANSWER:
# Why do we need to encode the Sex column?

# 14. Define Features and Target

In [ ]:
X = df.drop("Rings", axis=1)
y = df["Rings"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)

# 15. Identify Numerical and Categorical Features

In [ ]:
categorical_features = ["Sex"]

numerical_features = [
    "Length",
    "Diameter",
    "Height",
    "WholeWeight",
    "ShuckedWeight",
    "VisceraWeight",
    "ShellWeight"
]

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

# 16. Train-Test Split

We will use:

- **80%** data for training
- **20%** data for testing

The test set should not be used to train the model.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

# 17. Preprocessing Pipeline

`Sex` is categorical, so we use **One-Hot Encoding**.

The numerical features are passed through unchanged.

Using a pipeline helps keep preprocessing and model training together and reduces the risk of accidentally applying transformations incorrectly.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numerical_features)
    ]
)

# 🤖 Task 3 — Model 1: Linear Regression

Linear Regression is a useful baseline model for a regression problem.

### Questions

- What does the model predict?
- What does the prediction represent?
- Is a linear relationship necessarily sufficient for this problem?

In [ ]:
linear_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression())
    ]
)

linear_model.fit(X_train, y_train)

linear_predictions = linear_model.predict(X_test)

print("Linear Regression model trained successfully!")

# 18. Evaluate Linear Regression

We will use:

### MAE
Average absolute prediction error.

### MSE
Average squared prediction error.

### RMSE
Square root of MSE; expressed in the same units as the target.

### R²
Measures how much variation in the target is explained by the model.

In [ ]:
linear_mae = mean_absolute_error(y_test, linear_predictions)
linear_mse = mean_squared_error(y_test, linear_predictions)
linear_rmse = np.sqrt(linear_mse)
linear_r2 = r2_score(y_test, linear_predictions)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", linear_mae)
print("MSE :", linear_mse)
print("RMSE:", linear_rmse)
print("R²  :", linear_r2)

# 🌳 Task 4 — Model 2: Decision Tree Regressor

Now build a second model.

A Decision Tree can learn non-linear relationships between the input features and the target.

In [ ]:
tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", DecisionTreeRegressor(
            max_depth=6,
            random_state=42
        ))
    ]
)

tree_model.fit(X_train, y_train)

tree_predictions = tree_model.predict(X_test)

print("Decision Tree model trained successfully!")

# 19. Evaluate Decision Tree

In [ ]:
tree_mae = mean_absolute_error(y_test, tree_predictions)
tree_mse = mean_squared_error(y_test, tree_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_r2 = r2_score(y_test, tree_predictions)

print("Decision Tree Results")
print("---------------------")
print("MAE :", tree_mae)
print("MSE :", tree_mse)
print("RMSE:", tree_rmse)
print("R²  :", tree_r2)

# 🌲 Optional Bonus Model — Random Forest

If time permits, build a Random Forest model.

Random Forest combines many decision trees and can often capture more complex relationships.

In [ ]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        ))
    ]
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_predictions)
rf_mse = mean_squared_error(y_test, rf_predictions)
rf_rmse = np.sqrt(rf_mse)
rf_r2 = r2_score(y_test, rf_predictions)

print("Random Forest Results")
print("---------------------")
print("MAE :", rf_mae)
print("MSE :", rf_mse)
print("RMSE:", rf_rmse)
print("R²  :", rf_r2)

# 📊 20. Compare the Models

Create a single table containing the evaluation results.

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        linear_mae,
        tree_mae,
        rf_mae
    ],
    "MSE": [
        linear_mse,
        tree_mse,
        rf_mse
    ],
    "RMSE": [
        linear_rmse,
        tree_rmse,
        rf_rmse
    ],
    "R2": [
        linear_r2,
        tree_r2,
        rf_r2
    ]
})

results

# 🔍 Task 5 — Model Interpretation

Use your results table to answer:

1. Which model has the lowest MAE?
2. Which model has the lowest RMSE?
3. Which model has the highest R²?
4. Why might the models produce different results?
5. Which model would you investigate further?
6. Does a good test score guarantee that the model will work perfectly on every new abalone?

In [ ]:
# YOUR ANSWERS

# 1.
# 2.
# 3.
# 4.
# 5.
# 6.

# 21. Actual vs Predicted Values

A useful diagnostic is to compare the actual number of rings with the predicted number of rings.

We will visualize the Random Forest predictions.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, rf_predictions, alpha=0.5)

min_value = min(y_test.min(), rf_predictions.min())
max_value = max(y_test.max(), rf_predictions.max())

plt.plot([min_value, max_value], [min_value, max_value])
plt.xlabel("Actual Rings")
plt.ylabel("Predicted Rings")
plt.title("Actual vs Predicted Rings — Random Forest")
plt.show()

# 22. Prediction Error Analysis

Calculate the prediction error for each test observation.

**Error = Actual − Predicted**

In [ ]:
error_df = pd.DataFrame({
    "Actual_Rings": y_test.values,
    "Predicted_Rings": rf_predictions
})

error_df["Error"] = (
    error_df["Actual_Rings"] - error_df["Predicted_Rings"]
)

error_df.head(10)

# 🐚 Task 6 — Predict a New Abalone

The marine research organization provides a new abalone with these measurements:

| Feature | Value |
|---|---:|
| Sex | M |
| Length | 0.60 |
| Diameter | 0.48 |
| Height | 0.15 |
| WholeWeight | 1.20 |
| ShuckedWeight | 0.50 |
| VisceraWeight | 0.25 |
| ShellWeight | 0.35 |

Use your trained model to predict its number of rings.

In [ ]:
new_abalone = pd.DataFrame({
    "Sex": ["M"],
    "Length": [0.60],
    "Diameter": [0.48],
    "Height": [0.15],
    "WholeWeight": [1.20],
    "ShuckedWeight": [0.50],
    "VisceraWeight": [0.25],
    "ShellWeight": [0.35]
})

new_abalone

In [ ]:
predicted_rings = rf_model.predict(new_abalone)[0]
estimated_age = predicted_rings + 1.5

print("Predicted Rings:", round(predicted_rings, 2))
print("Estimated Age :", round(estimated_age, 2), "years")

# ⭐ Bonus Challenge — Feature Importance

Try to identify which physical measurements contribute most to the Random Forest model.

### Questions

1. Which feature is most important?
2. Which feature is least important?
3. Does feature importance prove that a feature causes age to increase?

> Remember: feature importance describes the model's use of a feature; it does not establish causation.

In [ ]:
# Extract the fitted preprocessing and model
fitted_preprocessor = rf_model.named_steps["preprocessor"]
fitted_rf = rf_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()

importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Importance": fitted_rf.feature_importances_
}).sort_values("Importance", ascending=False)

importance_df

In [ ]:
plt.figure(figsize=(9, 5))
plt.barh(
    importance_df["Feature"],
    importance_df["Importance"]
)
plt.gca().invert_yaxis()
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Random Forest Feature Importance")
plt.show()

# 🏁 Final Conclusion

Complete the following before submitting your notebook.

### Problem

What real-world problem did we solve?

### Problem Type

Why is this a regression problem?

### Data

What features were available?

### Preprocessing

What preprocessing did you perform and why?

### Models

Which models did you build?

### Evaluation

What were the MAE, RMSE and R² values?

### Model Comparison

How did the models differ?

### Prediction

What number of rings and estimated age did your model predict for the new abalone?

### Limitations

Give at least **two limitations** of your solution.

### Improvements

Give at least **two ways** the model could potentially be improved.

In [ ]:
# 📝 FINAL ANSWERS

print("1. Problem:")
# Write your answer

print("\n2. Problem Type:")
# Write your answer

print("\n3. Data & Features:")
# Write your answer

print("\n4. Preprocessing:")
# Write your answer

print("\n5. Models:")
# Write your answer

print("\n6. Evaluation:")
# Write your answer

print("\n7. Model Comparison:")
# Write your answer

print("\n8. Prediction:")
# Write your answer

print("\n9. Limitations:")
# Write your answer

print("\n10. Improvements:")
# Write your answer

# 🎤 Mini Presentation Challenge

Present your project in **3–5 minutes**.

### Your presentation should cover:

1. **Problem** — What are you predicting?
2. **Dataset** — Where did the data come from?
3. **EDA** — What did you discover?
4. **Preprocessing** — What did you do to the data?
5. **Models** — Which algorithms did you try?
6. **Results** — What metrics did you get?
7. **Prediction** — What did your model predict?
8. **Conclusion** — Can the model be useful in a real-world setting?

## Final Question

> **If a marine researcher gives you the measurements of a new abalone, can your model estimate its age accurately enough to be useful?**

Don't just show the score — **explain what the score means.**

# 📚 Reference

### Original Dataset

UCI Machine Learning Repository — Abalone Dataset

https://archive.ics.uci.edu/dataset/1/abalone

### Suggested Extensions

If you finish early, experiment with:

- Different `max_depth` values for Decision Tree
- Different numbers of trees in Random Forest
- Feature scaling
- Outlier analysis
- Cross-validation
- Hyperparameter tuning
- Additional regression algorithms
- Residual analysis